# Time Series Analysis

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
from scipy.signal import butter, filtfilt, find_peaks
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from timeit import default_timer as timer
from sklearn.preprocessing import StandardScaler

In [ ]:
FOLDER_PATH = Path("../../data/time-series")  

subdirs = sorted([d for d in FOLDER_PATH.iterdir() if d.is_dir()])

print(f"Total directories found: {len(subdirs)}\n")

subject_infos = []

# Extract subject_id and hadm_id (record) from directory names
for subdir in subdirs:
    # Split directory name: format is "subjectid_hadmid"
    parts = subdir.name.split('_')
    if len(parts) >= 2:
        subject_id = parts[0]
        hadm_id = parts[1]  # This is the record/admission ID
        subject_infos.append({
            'subject_id': subject_id,
            'record_name': hadm_id,
            'folder_name': subdir.name
        })

# Convert to DataFrame for easier manipulation
subject_infos_df = pd.DataFrame(subject_infos)

subject_infos_df.shape

In [ ]:
timeseries_dict = {}

for record in subject_infos_df['record_name'].unique():
    record_path = FOLDER_PATH / f"{subject_infos_df[subject_infos_df['record_name'] == record]['folder_name'].values[0]}" / record
    try:
        # Read the entire signal
        signal, fields = wfdb.rdsamp(str(record_path))
        fs = fields['fs']  # Sampling frequency
        
        # Use the actual length of the signal
        num_samples = signal.shape[0]
        num_channels = signal.shape[1]
        duration_seconds = num_samples / fs
        
        # Create time axis for the entire signal
        time_axis = np.arange(num_samples) / fs
        
        # Create a unique time series concatenating all channels sequentially
        # Flatten column-wise: Ch1 samples, then Ch2 samples, then Ch3 samples, etc.
        concatenated_signal = signal.flatten(order='F')  # 'F' for Fortran-style (column-major)
        
        total_length = concatenated_signal.shape[0]
        
        # Calculate start time (in seconds) for each channel in the concatenated signal
        signal_start_times = []
        for i in range(10):
            start_sample = i * num_samples
            start_time = start_sample / fs
            signal_start_times.append({
                'channel_index': i,
                'channel_name': fields['sig_name'][i],
                'start_sample': start_sample,
                'start_time_seconds': start_time,
                'end_sample': start_sample + num_samples - 1,
                'end_time_seconds': (start_sample + num_samples - 1) / fs
            })
        
        # Store in dictionary
        timeseries_dict[record] = {
            'signal': concatenated_signal,
            'fs': fs,
            'num_channels': num_channels,
            'samples_per_channel': num_samples,
            'total_length': total_length,
            'duration_seconds': duration_seconds,
            'signal_names': fields['sig_name'],
            'signal_start_times': signal_start_times
        }
        
        print(f"Record: {record}")
        print(f"  Channels: {num_channels}")
        print(f"  Samples per channel: {num_samples}")
        print(f"  Total concatenated length: {total_length} (= {num_channels} × {num_samples})")
        print(f"  Duration per channel: {duration_seconds:.2f} seconds")
        print(f"  Signal start times:")
        for st in signal_start_times[:3]:  # Show first 3 channels
            print(f"    {st['channel_name']}: starts at sample {st['start_sample']} (time: {st['start_time_seconds']:.2f}s)")
        if num_channels > 3:
            print(f"    ... and {num_channels - 3} more channels")
        print()

    except Exception as e:
        print(f"Could not read record {record}: {e}")

print(f"\nTotal records processed: {len(timeseries_dict)}")

## Example of ECG Signal

In [ ]:
# For the first subject, plot all the ECG signals in two columns of six plots
first_subject = subject_infos_df.iloc[0]
record_path = FOLDER_PATH / first_subject['folder_name'] / first_subject['record_name']
record = wfdb.rdrecord(str(record_path))
num_signals = record.p_signal.shape[1]
fig, axs = plt.subplots(6, 2, figsize=(15, 20))
for i in range(num_signals):
    ax = axs[i // 2, i % 2]
    ax.plot(record.p_signal[:, i])
    ax.set_title(f'Signal {i+1}')
plt.tight_layout()
plt.show()

# Time Series Understanding

Given a time series, it is interesting to understand the behaviour of the it; to do so we can check for some particular informations:
- **Trends**
- **Seasonality**
- **Cyclicity**

## Noise removal

### Rolling window approach

The first idea to remove noise in the time series is to use a rolling window (with a defeault value of 5), and smooth the curve. 

In [ ]:
# Remove noise in the time series by applying a rolling window average filter
def moving_average(signal, window_size=5):
    """
    Apply moving average filter to a numpy array signal.
    
    Parameters:
    -----------
    signal : numpy.ndarray
        Input signal
    window_size : int
        Size of the moving average window
        
    Returns:
    --------
    pandas.Series
        Filtered signal with NaN at edges
    """
    # Convert to pandas Series for rolling window operations
    signal_series = pd.Series(signal)
    
    # Standardize: (x - mean) / std
    standardized = (signal_series - signal_series.mean()) / signal_series.std()
    
    # Apply rolling window average
    filtered = standardized.rolling(window=window_size, center=True).mean()
    
    return filtered

def moving_std(signal, window_size=5):
    """
    Apply moving standard deviation filter to a numpy array signal.
    
    Parameters:
    -----------
    signal : numpy.ndarray
        Input signal
    window_size : int
        Size of the moving std window
        
    Returns:
    --------
    pandas.Series
        Filtered signal with NaN at edges
    """
    # Convert to pandas Series for rolling window operations
    signal_series = pd.Series(signal)
    
    # Standardize: (x - mean) / std
    standardized = (signal_series - signal_series.mean()) / signal_series.std()
    
    # Apply rolling window standard deviation
    filtered = standardized.rolling(window=window_size, center=True).std()
    
    return filtered


### Low/High-Pass and Band-Pass filters

Another option is to use *Low-Pass*, *High-Pass* and *Band-Pass* filters to remove the noise on ECG signals.
- **Low-Pass filter**: A Low-Pass filter (LPF) is a filter that passes signals with a frequency lower than a selected cutoff frequency and attenuates signals with frequencies higher than the cutoff frequency.
- **High-Pass filter**: A High-Pass filter (HPF) is an electronic filter that passes signals with a frequency higher than a certain cutoff frequency and attenuates signals with frequencies lower than the cutoff frequency.
- **Band-Pass filter**: A Band-Pass filter allows through components in a specified band of frequencies, called its passband but blocks components with frequencies above or below this band. This contrasts with a high-pass filter, which allows through components with frequencies above a specific frequency, and a low-pass filter, which allows through components with frequencies below a specific frequency.

In [ ]:
def low_pass_filter(signal, fs, cutoff=30, order=5):
    """
    Apply a low-pass Butterworth filter to the signal.

    Parameters:
    signal : numpy.ndarray
        Input signal
    fs : float
        Sampling frequency
    cutoff : float
        Cutoff frequency of the filter
    order : int
        Order of the filter
    Returns:
    numpy.ndarray
        Filtered signal
    """
    # Calculate the Nyquist frequency
    nyquist = 0.5 * fs

    # Normalized cutoff frequency
    normal_cutoff = cutoff / nyquist

    # Get the filter coefficients
    b, a = butter(order, normal_cutoff, btype='low', analog=False)

    # Apply the filter
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal

def high_pass_filter(signal, fs, cutoff=0.5, order=5):
    """
    Apply a high-pass Butterworth filter to the signal.

    Parameters:
    signal : numpy.ndarray
        Input signal
    fs : float
        Sampling frequency
    cutoff : float
        Cutoff frequency of the filter
    order : int
        Order of the filter
    Returns:
    numpy.ndarray
        Filtered signal
    """
    # Calculate the Nyquist frequency
    nyquist = 0.5 * fs

    # Normalized cutoff frequency
    normal_cutoff = cutoff / nyquist

    # Get the filter coefficients
    b, a = butter(order, normal_cutoff, btype='high', analog=False)

    # Apply the filter
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal

def band_pass_filter(signal, fs, lowcut=0.5, highcut=30, order=5):
    """
    Apply a band-pass Butterworth filter to the signal.

    Parameters:
    signal : numpy.ndarray
        Input signal
    fs : float
        Sampling frequency
    lowcut : float
        Low cutoff frequency of the filter
    highcut : float
        High cutoff frequency of the filter
    order : int
        Order of the filter
    Returns:
    numpy.ndarray
        Filtered signal
    """
    # Calculate the Nyquist frequency
    nyquist = 0.5 * fs

    # Normalized cutoff frequencies
    low = lowcut / nyquist
    high = highcut / nyquist

    # Get the filter coefficients
    b, a = butter(order, [low, high], btype='band')

    # Apply the filter
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal


### Triple Exponential Smoothing (Holt-Winters method)

In [ ]:
def hw_method(series, seasonal_periods, trend='add', seasonal='add'):
    """
    Apply Holt-Winters Exponential Smoothing to a time series.
    
    Parameters:
    -----------
    series : array-like
        Input time series data
    seasonal_periods : int
        Number of periods in a complete seasonal cycle
    trend : str
        Type of trend component ('add' or 'mul')
    seasonal : str
        Type of seasonal component ('add' or 'mul')
    
    Returns:
    --------
    fitted_model : ExponentialSmoothing results object
        Fitted Holt-Winters model
    """
    model = ExponentialSmoothing(
        series,
        trend=trend,
        seasonal=seasonal,
        seasonal_periods=seasonal_periods
    )
    fit_model = model.fit()
    return fit_model


def plot_hw_decomposition(series, fitted):
    # Extract Holt-Winters internal components
    level = fitted.level
    trend = fitted.trend
    seasonal = fitted.season
    fittedvalues = fitted.fittedvalues
    
    # Remainder (error)
    remainder = series - fittedvalues

    # ----- Plot -----
    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True, dpi = 300)
    
    # 1. Original time series
    axes[0].plot(series, color="black")
    axes[0].set_title("Time series (data)")
    
    # 2. Seasonal component
    axes[1].plot(seasonal, color="teal")
    axes[1].set_title("Seasonal patterns")
    
    # 3. Trend component
    axes[2].plot(trend, color="red")
    axes[2].set_title("General tendencies (trend)")
    
    # 4. Remainder
    axes[3].plot(remainder, color="green")
    axes[3].set_title("Error / Remainder")

    plt.xlabel("Time")
    plt.tight_layout()
    plt.show()

In [ ]:
def perform_holt_winters_grid_search(series, fundamental_lag, train_ratio=0.9, verbose=True):
    """
    Perform Holt-Winters grid search on a time series.
    
    Parameters:
    -----------
    series : pandas.Series or numpy.ndarray
        Input time series to fit
    fundamental_lag : int
        Fundamental lag from ACF analysis (used to generate seasonal candidates)
    train_ratio : float
        Train/test split ratio (default: 0.9 for 90/10 split)
    verbose : bool
        Whether to print progress and results (default: True)
    
    Returns:
    --------
    dict : Dictionary containing:
        - 'best_model': The best fitted ExponentialSmoothing model
        - 'best_params': Dictionary of best parameters
        - 'results_df': DataFrame with all search results sorted by validation SSE
        - 'train_set': Training data used
        - 'test_set': Test data used
        - 'metrics': Dictionary with search metrics (time, convergences, etc.)
    """
    from sklearn.model_selection import ParameterGrid
    import warnings
    import time
    
    # Convert to pandas Series if needed
    if not isinstance(series, pd.Series):
        series = pd.Series(series, dtype=float)
    else:
        series = series.copy()
    
    # Train/test split
    split_idx = int(train_ratio * len(series))
    train_set = series.iloc[:split_idx]
    test_set = series.iloc[split_idx:]
    
    if verbose:
        print(f"Series length: {len(series)}")
        print(f"Train set: {len(train_set)} samples ({train_ratio:.1%})")
        print(f"Test set: {len(test_set)} samples ({1-train_ratio:.1%})")
        print(f"Fundamental lag: {fundamental_lag}\n")
    
    # Generate seasonal candidates around fundamental lag
    seasonal_candidates = sorted({
        max(2, int(fundamental_lag + shift))
        for shift in range(-3, 12)
    })
    
    # Parameter grid
    params_grid = {
        'trend': ['add', 'mul', None],
        'seasonal': ['add', 'mul', None],
        'damped_trend': [False, True],
        'seasonal_periods': seasonal_candidates,
        'use_boxcox': ['log', None],
    }
    
    total_combinations = len(list(ParameterGrid(params_grid)))
    if verbose:
        print(f"Seasonal candidates: {len(seasonal_candidates)} (range: {seasonal_candidates[0]}-{seasonal_candidates[-1]})")
        print(f"Total parameter combinations to evaluate: {total_combinations}\n")
    
    # Grid search
    search_results = []
    best_model = None
    best_score = np.inf
    best_params = None
    failed_count = 0
    start_time = time.time()
    
    for idx, params in enumerate(ParameterGrid(params_grid), 1):
        # Skip invalid combos early
        if params['trend'] is None and params['damped_trend']:
            continue
        if params['seasonal'] is None:
            params['seasonal_periods'] = None
        elif params['seasonal_periods'] < 2:
            continue
        
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=ConvergenceWarning)
            try:
                model = ExponentialSmoothing(
                    train_set,
                    trend=params['trend'],
                    seasonal=params['seasonal'],
                    seasonal_periods=params['seasonal_periods'],
                    damped_trend=params['damped_trend'],
                    initialization_method='estimated'
                )
                fitted = model.fit(
                    optimized=True,
                    use_boxcox=params['use_boxcox']
                )
                
                # Forecast and compute validation SSE
                val_forecast = fitted.forecast(steps=len(test_set))
                val_sse = float(np.sum((val_forecast.values - test_set.values) ** 2))
                
                # Handle MAPE calculation safely
                try:
                    mape = np.mean(np.abs((test_set.values - val_forecast.values) / test_set.values)) * 100
                except:
                    mape = np.nan
                
                metrics = {
                    'trend': params['trend'],
                    'seasonal': params['seasonal'],
                    'seasonal_periods': params['seasonal_periods'],
                    'damped_trend': params['damped_trend'],
                    'use_boxcox': params['use_boxcox'],
                    'train_sse': fitted.sse,
                    'train_aic': fitted.aic,
                    'val_sse': val_sse,
                    'mape': mape,
                }
                search_results.append((metrics, fitted))
                
                if val_sse < best_score:
                    best_score = val_sse
                    best_model = fitted
                    best_params = params.copy()
                    if verbose:
                        print(f"  ✓ Iteration {idx}: Val SSE = {val_sse:.4f} (Best so far)")
                    
            except Exception as e:
                failed_count += 1
                continue
    
    elapsed_time = time.time() - start_time
    
    if not search_results:
        raise RuntimeError("No Holt-Winters configuration converged.")
    
    # Sort results by validation SSE
    results_df = pd.DataFrame([m for m, _ in search_results]).sort_values('val_sse')
    
    if verbose:
        print(f"\n{'='*90}")
        print(f"Grid Search Summary")
        print(f"{'='*90}")
        print(f"Total combinations evaluated: {len(search_results)}")
        print(f"Failed convergences: {failed_count}")
        print(f"Success rate: {len(search_results)/(len(search_results)+failed_count)*100:.1f}%")
        print(f"Time elapsed: {elapsed_time:.2f} seconds")
        print(f"Average time per model: {elapsed_time/len(search_results)*1000:.2f} ms")
        print(f"{'='*90}\n")
        
        print("Top 10 Best Models:")
        display_cols = ['trend', 'seasonal', 'seasonal_periods', 'damped_trend', 'use_boxcox', 'val_sse', 'mape']
        print(results_df.head(10)[display_cols].to_string())
        
        print(f"\n{'='*90}")
        print(f"Best Model Parameters:")
        print(f"{'='*90}")
        best_row = results_df.iloc[0]
        print(f"  Trend:           {best_row['trend']}")
        print(f"  Seasonal:        {best_row['seasonal']}")
        print(f"  Seasonal Period: {int(best_row['seasonal_periods']) if pd.notna(best_row['seasonal_periods']) else 'N/A'}")
        print(f"  Damped Trend:    {best_row['damped_trend']}")
        print(f"  Box-Cox:         {best_row['use_boxcox']}")
        print(f"  Train SSE:       {best_row['train_sse']:.4f}")
        print(f"  Train AIC:       {best_row['train_aic']:.4f}")
        print(f"  Validation SSE:  {best_row['val_sse']:.4f}")
        print(f"  MAPE:            {best_row['mape']:.2f}%")
        print(f"{'='*90}\n")
    
    return {
        'best_model': best_model,
        'best_params': best_params,
        'results_df': results_df,
        'train_set': train_set,
        'test_set': test_set,
        'metrics': {
            'total_evaluated': len(search_results),
            'failed_convergences': failed_count,
            'success_rate': len(search_results)/(len(search_results)+failed_count),
            'elapsed_time': elapsed_time,
            'best_val_sse': best_score,
        }
    }

### Pan-Tompkins algorithm

In [ ]:
def pt_algorithm(signal, fs):
    """
    Apply the Pan-Tompkins algorithm to detect R-peaks in ECG signals.
    
    Parameters:
    -----------
    signal : numpy.ndarray
        Input ECG signal
    fs : float
        Sampling frequency
        
    Returns:
    --------
    r_peaks : numpy.ndarray
        Indices of detected R-peaks in the signal
    """
    # Step 1: Low-pass filter (0.5-30 Hz)
    lp_filtered = low_pass_filter(signal, fs, cutoff=30, order=5)

    hp_filtered = high_pass_filter(lp_filtered, fs, cutoff=0.5, order=5)
    
    # Step 2: Derivative filter
    derivative_signal = np.diff(hp_filtered)
    
    # Step 3: Squaring
    squared_signal = np.square(derivative_signal)
    
    # Step 4: Moving window integration
    window_size = int(0.150 * fs)  # 150 ms window
    integrated_signal = moving_average(squared_signal, window_size=window_size).fillna(0).values
    
    return integrated_signal

In [ ]:
def QRSpeaks(signal, fs):
    """
    Detect QRS peaks (R-peaks) in Pan-Tompkins processed signal.
    
    Parameters:
    -----------
    signal : numpy.ndarray
        Integrated signal from Pan-Tompkins algorithm
    fs : float
        Sampling frequency
        
    Returns:
    --------
    peaks : numpy.ndarray
        Indices of detected R-peaks
    """
    # Use find_peaks from scipy.signal (already imported)
    peaks, _ = find_peaks(signal, height=np.mean(signal), distance=round(fs*0.200))
    return peaks

## Time series data quality

In [ ]:
# For each subject, take only the first channel's data for analysis
subj_dict = {}
for record, data in timeseries_dict.items():
    fs = data['fs']
    num_samples = data['samples_per_channel']
    first_channel_signal = data['signal'][:num_samples]  # First channel data
    timeseries_dict[record]['first_channel_signal'] = first_channel_signal
    subj_dict[record] = {
        'signal': first_channel_signal,
        'fs': fs,
        'num_samples': num_samples
    }

### Noise cancellation approach 1: Using the rolling window to clean the time series

In [ ]:
# Example plotting the differences in filtering results
subject_id = list(subj_dict.keys())[0]
original_signal = subj_dict[subject_id]['signal']
ma_filtered_signal = moving_average(original_signal, window_size=5).fillna(0).values

### Noise cancellation approach 2: Using the Band-Pass filter to cancel the noise in the time series

In [ ]:
scaler = StandardScaler()
# Reshape to 2D for scaler, then flatten back to 1D
scaled_signal = scaler.fit_transform(original_signal.reshape(-1, 1)).flatten()

bp_filtered = band_pass_filter(scaled_signal, subj_dict[subject_id]['fs'], lowcut=0.5, highcut=30, order=3)

#### Comparison of the two methods

In [ ]:
# Calculate the differences between the filtered and the original
diff_ma = ma_filtered_signal - original_signal
diff_bp = bp_filtered - scaled_signal

print(f"Difference MA Filter - Mean: {np.mean(diff_ma):.4f}, Std: {np.std(diff_ma):.4f}")
print(f"Difference BP Filter - Mean: {np.mean(diff_bp):.4f}, Std: {np.std(diff_bp):.4f}")


fig, axes = plt.subplots(3, 1, figsize=(14, 10), dpi = 300)

# Plot 1: Original Signal
axes[0].plot(original_signal, label='Original Signal', color='blue', alpha=0.7)
axes[0].set_title(f'Original Signal - Subject {subject_id}')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# Plot 2: Moving Average Filtered Signal
axes[1].plot(ma_filtered_signal, label='Moving Average Filtered', color='red', alpha=0.7)
axes[1].set_title(f'Moving Average Filtered - Subject {subject_id}')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

axes[2].plot(bp_filtered, label='Band-Pass Filtered', color='green', alpha=0.7)
axes[2].set_title(f'Band-Pass Filtered - Subject {subject_id}')
axes[2].set_xlabel('Sample Index')
axes[2].set_ylabel('Amplitude')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
# Create unified legend outside the plot area
handles = [axes[0].get_lines()[0], axes[1].get_lines()[0], axes[2].get_lines()[0]]
labels = ['Original Signal', 'Moving Average Filtered', 'Band-Pass Filtered']
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=True)
plt.show()

In [ ]:
# For all the subjects, use the band-pass filter and save the filtered signals
for subject_id, data in subj_dict.items():
    original_signal = data['signal']
    fs = data['fs']
    
    # Standardize the signal
    scaler = StandardScaler()
    scaled_signal = scaler.fit_transform(original_signal.reshape(-1, 1)).flatten()
    
    # Apply band-pass filter
    bp_filtered = band_pass_filter(scaled_signal, fs, lowcut=0.5, highcut=30, order=3)
    
    # Save back to the dictionary
    subj_dict[subject_id]['band_pass_filtered'] = bp_filtered

#### Peak detection using Pan-Tompkins algorithm

In [ ]:
# Run the Pan-Tompkins algorithm on the band-pass filtered signals to detect R-peaks
for subject_id, data in subj_dict.items():
    bp_filtered_signal = data['band_pass_filtered']
    fs = data['fs']
    
    # Apply Pan-Tompkins algorithm
    integrated_signal = pt_algorithm(bp_filtered_signal, fs)
    
    # Detect R-peaks using the QRSpeaks function
    r_peaks = QRSpeaks(integrated_signal, fs)
    
    # Save R-peaks indices
    subj_dict[subject_id]['r_peaks'] = r_peaks

In [ ]:
# Get the seasonal period using the Pan-Tompkins detected R-peaks
for subject_id, data in subj_dict.items():
    r_peaks = data['r_peaks']
    fs = data['fs']
    
    if len(r_peaks) < 2:
        print(f"Not enough R-peaks detected for subject {subject_id} to calculate seasonal period.")
        continue
    
    # Calculate RR intervals in samples
    rr_intervals_samples = np.diff(r_peaks)
    
    # Convert to seconds
    rr_intervals_seconds = rr_intervals_samples / fs
    
    # Calculate average heart rate (beats per minute)
    avg_rr_interval = np.mean(rr_intervals_seconds)
    avg_heart_rate_bpm = 60 / avg_rr_interval
    
    # Estimate seasonal period in samples (average number of samples per beat)
    seasonal_period_samples = int(np.round(fs * avg_rr_interval))

    subj_dict[subject_id]['avg_heart_rate_bpm'] = avg_heart_rate_bpm
    subj_dict[subject_id]['seasonal_period_samples'] = seasonal_period_samples


#### Running the Holt-Winters method

In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# Run the Holt-Winters method using the seasonal periods obtained from R-peak detection
hw_results = {}
for subject_id, data in subj_dict.items():
    seasonal_period = data.get('seasonal_period_samples', None)
    if seasonal_period is None:
        print(f"Skipping subject {subject_id} due to missing seasonal period.")
        continue
    
    signal = data['band_pass_filtered']
    
    # Validate seasonal_periods is less than series length
    if seasonal_period >= len(signal):
        seasonal_period = max(2, len(signal) // 3)
    
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=ConvergenceWarning)
            hw_result = hw_method(
                series=signal,
                seasonal_periods=seasonal_period,
                trend='add',
                seasonal='add'
            )
        hw_results[subject_id] = hw_result
    except Exception as e:
        print(f"Failed Holt-Winters for Subject {subject_id}: {e}")